# Advanced Data Imputation and Encoding in Scikit-Learn

This tutorial covers advanced strategies for handling missing values and encoding categorical features using Scikit-Learn. You will learn how to apply univariate and multivariate imputation, plus how to perform target encoding safely.

## Setup

Import the required libraries and build a synthetic dataset with missing numerical values and categorical features.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

random_state = 42
rng = np.random.RandomState(random_state)

data = pd.DataFrame({
    'age': [29, 35, np.nan, 50, 40, 28, np.nan, 52, 41, 33, 30, np.nan, 48, 36, 39],
    'income': [65000, 72000, 58000, np.nan, 81000, 52000, 64000, 90000, np.nan, 71000, 68000, 63000, 77000, np.nan, 59000],
    'city': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'A', 'B', 'C', 'A', 'B', 'C', 'A', 'B'],
    'color': ['red', 'blue', 'green', 'blue', 'red', 'green', 'red', 'blue', 'green', 'red', 'blue', 'green', 'red', 'blue', 'green'],
    'purchase': [1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0],
})

data.loc[[1, 7], 'color'] = np.nan

print('Synthetic dataset sample:')
print(data.head(8))

feature_columns = ['age', 'income', 'city', 'color']
target_column = 'purchase'

X = data[feature_columns]
y = data[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=random_state, stratify=y
)

print('Train set shape:', X_train.shape)
print('Test set shape:', X_test.shape)

Synthetic dataset sample:
    age   income city  color  purchase
0  29.0  65000.0    A    red         1
1  35.0  72000.0    B    NaN         0
2   NaN  58000.0    A  green         1
3  50.0      NaN    C   blue         1
4  40.0  81000.0    B    red         0
5  28.0  52000.0    A  green         0
6   NaN  64000.0    C    red         1
7  52.0  90000.0    A    NaN         1
Train set shape: (11, 4)
Test set shape: (4, 4)


## Univariate Imputation

Use `SimpleImputer` for single-variable strategies. `mean` and `median` are common for numeric features, while `most_frequent` works well for categorical values.

In [ ]:
from sklearn.impute import SimpleImputer

numeric_train = X_train[['age', 'income']].copy()
categorical_train = X_train[['city', 'color']].copy()

mean_imputer = SimpleImputer(strategy='mean')
median_imputer = SimpleImputer(strategy='median')
mode_imputer = SimpleImputer(strategy='most_frequent')

age_mean = mean_imputer.fit_transform(numeric_train[['age']])
income_median = median_imputer.fit_transform(numeric_train[['income']])
city_mode = mode_imputer.fit_transform(categorical_train[['city']])

print('Age imputation with mean:')
print(age_mean.ravel())
print('Income imputation with median:')
print(income_median.ravel())
print('City imputation with most frequent:')
print(city_mode.ravel())

Age imputation with mean:
[35.    29.    48.    40.    50.    33.    36.625 36.625 28.    30.
 36.625]
Income imputation with median:
[72000. 65000. 77000. 81000. 66500. 71000. 63000. 58000. 52000. 68000.
 64000.]
City imputation with most frequent:
['B' 'A' 'C' 'B' 'C' 'C' 'B' 'A' 'A' 'A' 'C']


## Multivariate Imputation

Multivariate imputers infer missing values using relationships between features. `IterativeImputer` models each feature as a function of others, while `KNNImputer` uses nearby samples.

In [ ]:
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.linear_model import BayesianRidge

numeric_features = ['age', 'income']
numeric_train = X_train[numeric_features].copy()
numeric_test = X_test[numeric_features].copy()

iterative_imputer = IterativeImputer(
    estimator=BayesianRidge(), random_state=random_state, max_iter=10
)
knn_imputer = KNNImputer(n_neighbors=3)

numeric_train_iter = iterative_imputer.fit_transform(numeric_train)
numeric_test_iter = iterative_imputer.transform(numeric_test)

numeric_train_knn = knn_imputer.fit_transform(numeric_train)
numeric_test_knn = knn_imputer.transform(numeric_test)

numeric_train_iter_df = pd.DataFrame(
    numeric_train_iter, columns=numeric_features, index=numeric_train.index
)
numeric_test_iter_df = pd.DataFrame(
    numeric_test_iter, columns=numeric_features, index=numeric_test.index
)
numeric_train_knn_df = pd.DataFrame(
    numeric_train_knn, columns=numeric_features, index=numeric_train.index
)
numeric_test_knn_df = pd.DataFrame(
    numeric_test_knn, columns=numeric_features, index=numeric_test.index
)

print('IterativeImputer on train:')
print(numeric_train_iter_df.head())
print('KNNImputer on train:')
print(numeric_train_knn_df.head())

IterativeImputer on train:
     age        income
1   35.0  72000.000000
0   29.0  65000.000000
12  48.0  77000.000000
4   40.0  81000.000000
3   50.0  84380.684332
KNNImputer on train:
     age        income
1   35.0  72000.000000
0   29.0  65000.000000
12  48.0  77000.000000
4   40.0  81000.000000
3   50.0  76666.666667


## Handling the Target with Target Encoding

Target encoding replaces each categorical level with the mean target value for that level. It is useful for high-cardinality categories and can capture information that ordinal or one-hot encoders do not.

In [10]:
from sklearn.preprocessing import TargetEncoder

categorical_features = ['city', 'color']
X_train_cat = X_train[categorical_features].copy()
X_test_cat = X_test[categorical_features].copy()

target_encoder = TargetEncoder(cv=3)
X_train_encoded = target_encoder.fit_transform(X_train_cat, y_train)
X_test_encoded = target_encoder.transform(X_test_cat)

print('Encoded training categories:')
X_train_encoded = pd.DataFrame(
    X_train_encoded, columns=categorical_features, index=X_train_cat.index
)
X_test_encoded = pd.DataFrame(
    X_test_encoded, columns=categorical_features, index=X_test_cat.index
)
print(X_train_encoded.head())
print('Encoded test categories:')
print(X_test_encoded.head())

Encoded training categories:
        city     color
1   0.000000  0.500000
0   0.524138  1.000000
12  1.000000  0.721311
4   0.000000  1.000000
3   1.000000  0.000000
Encoded test categories:
        city     color
7   0.509151  0.000000
13  0.509151  0.515235
8   0.000000  0.382129
14  0.000000  0.382129


## Best Practices

Always fit imputation and encoding transformers on the training set only. Then apply `transform` to the test set to prevent data leakage and preserve realistic evaluation.

In [13]:
numeric_imputer = SimpleImputer(strategy='mean')
target_encoder = TargetEncoder(cv=3)

numeric_imputer.fit(numeric_train)
X_train_numeric_imputed = numeric_imputer.transform(numeric_train)
X_test_numeric_imputed = numeric_imputer.transform(numeric_test)

target_encoder.fit(X_train_cat, y_train)
X_train_cat_encoded = target_encoder.transform(X_train_cat)
X_test_cat_encoded = target_encoder.transform(X_test_cat)

print('Train numeric imputed shape:', X_train_numeric_imputed.shape)
print('Test numeric imputed shape:', X_test_numeric_imputed.shape)
print('Train encoded categories:')
X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded, columns=categorical_features, index=X_train_cat.index
)
X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded, columns=categorical_features, index=X_test_cat.index
)
print(X_train_cat_encoded.head())
print('Test encoded categories:')
print(X_test_cat_encoded.head())

Train numeric imputed shape: (11, 2)
Test numeric imputed shape: (4, 2)
Train encoded categories:
        city     color
1   0.000000  0.000000
0   0.509151  0.770902
12  1.000000  0.770902
4   0.000000  0.770902
3   1.000000  0.515235
Test encoded categories:
        city     color
7   0.509151  0.000000
13  0.509151  0.515235
8   0.000000  0.382129
14  0.000000  0.382129
